# 04 - Risk Measures: VaR, CVaR & Backtesting

**Project:** ARMA-GARCH Beta Risk Model  
**Author:** Amos Anderson  
**Purpose:** Compute daily Value at Risk (VaR) and Conditional Value at Risk
(CVaR) forecasts using fitted NIG parameters and GARCH $\sigma $ forecasts.
Backtest predictions against actual returns, count exceedances, and
compute binomial p-values with green/red/blue color coding.

Confidence levels: 95% and 99%.  
Assessment window: 1,000 trading days (2021-11-05 to 2025-12-30).

In [1]:
# Imports 

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from scipy import stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "iframe"

import warnings
warnings.filterwarnings("ignore")

from src.data_utils import load_processed, save_processed
from src.nig import NIGParams, compute_var, compute_cvar

print("Imports OK")

Imports OK


In [3]:
# Load all data

innovations     = load_processed("innovations.parquet")
sigma_forecasts = load_processed("sigma_forecasts.parquet")
nig_params_df   = load_processed("nig_params.parquet")

TICKERS = list(innovations.columns)

# Reconstruct NIGParams objects from saved DataFrame
nig_params = {}
for ticker in TICKERS:
    row = nig_params_df.loc[ticker]
    nig_params[ticker] = NIGParams(
        alpha = row["alpha"],
        beta  = row["beta"],
        mu    = row["mu"],
        delta = row["delta"],
    )

# Load actual returns aligned to assessment window
all_results = {}
for ticker in TICKERS:
    safe_name = ticker.replace("^", "").replace("=", "_")
    all_results[ticker] = load_processed(f"rolling_{safe_name}.parquet")

print(f"\nLoaded {len(TICKERS)} assets")
print(f"Assessment window: {innovations.index[0].date()} to {innovations.index[-1].date()}")
print(f"\nNIG parameters:\n{nig_params_df.round(4).to_string()}")

Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\innovations.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\sigma_forecasts.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\nig_params.parquet  shape=(5, 4)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\rolling_AAPL.parquet  shape=(1000, 3)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arm

----

## Risk measure definitions

**Value at Risk (VaR) at level $\alpha$:**  
The loss threshold not exceeded with probability $\alpha $.  
$$\text{VaR}_{99} = -\text{Q\_NIG}(0.01) \times \sigma_{\text{forecast}}$$
Interpretation: there is a 1% chance tomorrow's loss exceeds this number.

**Conditional VaR (CVaR) at level $\alpha $, also called Expected Shortfall:**  
The expected loss conditional on being in the worst $(1-\alpha)$% of outcomes.  
$$\text{CVaR}_{99} = E[r | r < \text{VaR}_{99}]$$
Always more negative than VaR. Required by Basel III for bank capital adequacy.

**Exceedance:**  
An actual return that falls below the VaR threshold on that day.

With a correctly calibrated model, exceedances occur at exactly rate $(1-\alpha)$. \
$\text{VaR}_{99} \rightarrow$ expect ~10 exceedances per 1,000 predictions (acceptable range: 6-15). \
$\text{VaR}_{95} \rightarrow$ expect ~50 exceedances per 1,000 predictions (acceptable range: 37-63).

**Binomial p-value:**  
Probability of observing exactly k exceedances if the model is correct.
- p $\geq $ 0.05 $\rightarrow$ model passes (green).
  The number of observed exceedances is within the expected range. The model is statistically "accurate."
- p < 0.05 with too many exceedances $\rightarrow$ risk understated, model fails (red).
  That is, we observed significantly more breaches than predicted. This is the most dangerous scenario for a firm because the VaR is not capturing the actual risk exposure, leading to insufficient capital buffers.
- p < 0.05 with too few exceedances $\rightarrow$ risk overstated, model is conservative (blue).
  This implies that we observed significantly fewer breaches than predicted. While the model is "inaccurate" from a statistical standpoint, it is "safe" or "conservative." In banking (e.g., Basel requirements), this usually doesn't trigger penalties, but it suggests the model is inefficient and holding too much capita

In [4]:
# Implement src/assessment.py

import importlib
import src.assessment as _assess
importlib.reload(_assess)
from src.assessment import (
    count_exceedances,
    binomial_pvalue,
    pvalue_color,
    christoffersen_test,
    anderson_darling,
)
print("src.assessment imported OK")

src.assessment imported OK


In [5]:
# Compute VaR and CVaR for all assets

from tqdm import tqdm

LEVELS = [0.99, 0.95]
var_results  = {level: {} for level in LEVELS}
cvar_results = {level: {} for level in LEVELS}

for ticker in TICKERS:
    p      = nig_params[ticker]
    sigmas = sigma_forecasts[ticker].values
    print(f"Computing VaR/CVaR -->  {ticker}...", end=" ")

    for level in LEVELS:
        var_series  = np.array([compute_var(s, p, level)  for s in sigmas])
        cvar_series = np.array([compute_cvar(s, p, level) for s in sigmas])
        var_results[level][ticker]  = var_series
        cvar_results[level][ticker] = cvar_series

    print("done")

print("\nVaR/CVaR computation complete.")

Computing VaR/CVaR -->  AAPL... done
Computing VaR/CVaR -->  EURUSD=X... done
Computing VaR/CVaR -->  GLD... done
Computing VaR/CVaR -->  TIP... done
Computing VaR/CVaR -->  ^GSPC... done

VaR/CVaR computation complete.


In [6]:
# Count exceedances and compute p-values

from src.assessment import count_exceedances, binomial_pvalue, pvalue_color

N = 1000  # assessment window size
results_table = []

for ticker in TICKERS:
    actual = all_results[ticker]["return"].values

    for level in LEVELS:
        var_s    = var_results[level][ticker]
        exceed   = count_exceedances(actual, var_s)
        expected = N * (1 - level)
        pval     = binomial_pvalue(N, exceed, level)
        color    = pvalue_color(pval, exceed, expected)

        results_table.append({
            "Ticker":      ticker,
            "VaR level":   f"{int(level*100)}%",
            "Expected":    int(expected),
            "Exceedances": exceed,
            "Excess":      exceed - int(expected),
            "p-value":     round(pval, 4),
            "Result":      color.upper(),
        })

results_df = pd.DataFrame(results_table)
print("Exceedance Results\n")
print(results_df.to_string(index=False))

Exceedance Results

  Ticker VaR level  Expected  Exceedances  Excess  p-value Result
    AAPL       99%        10            7      -3   0.4264  GREEN
    AAPL       95%        50           48      -2   0.8278  GREEN
EURUSD=X       99%        10            8      -2   0.6343  GREEN
EURUSD=X       95%        50           54       4   0.5611  GREEN
     GLD       99%        10           10       0   1.0000  GREEN
     GLD       95%        50           53       3   0.6629  GREEN
     TIP       99%        10           11       1   0.7486  GREEN
     TIP       95%        50           51       1   0.8845  GREEN
   ^GSPC       99%        10           11       1   0.7486  GREEN
   ^GSPC       95%        50           51       1   0.8845  GREEN


In [9]:
# Color-coded p-value table with Plotly

color_map = {"GREEN": "#2dc653", "RED": "#e63946", "BLUE": "#457b9d"}
font_map  = {"GREEN": "#144a23", "RED": "#5c0a10", "BLUE": "#0d2b40"}

var99_df = results_df[results_df["VaR level"] == "99%"].set_index("Ticker")
var95_df = results_df[results_df["VaR level"] == "95%"].set_index("Ticker")

def make_pvalue_table(df, title):
    fill_colors = [
        [color_map[r] for r in df["Result"]],
    ]
    font_colors = [
        [font_map[r] for r in df["Result"]],
    ]

    fig = go.Figure(data=[go.Table(
        header=dict(
            values=["Ticker", "Expected", "Exceedances",
                    "Excess", "p-value", "Result"],
            fill_color="#2c3e50",
            font=dict(color="white", size=12),
            align="center",
            height=32,
        ),
        cells=dict(
            values=[
                df.index.tolist(),
                df["Expected"].tolist(),
                df["Exceedances"].tolist(),
                df["Excess"].tolist(),
                df["p-value"].tolist(),
                df["Result"].tolist(),
            ],
            fill_color=[
                ["#f8f9fa"] * len(df),
                ["#f8f9fa"] * len(df),
                ["#f8f9fa"] * len(df),
                ["#f8f9fa"] * len(df),
                ["#f8f9fa"] * len(df),
                [color_map[r] for r in df["Result"]],
            ],
            font=dict(
                color=[
                    ["#2c3e50"] * len(df),
                    ["#2c3e50"] * len(df),
                    ["#2c3e50"] * len(df),
                    ["#2c3e50"] * len(df),
                    ["#2c3e50"] * len(df),
                    [font_map[r] for r in df["Result"]],
                ],
                size=12,
            ),
            align="center",
            height=28,
        ),
    )])

    fig.update_layout(
        title=title,
        template="plotly_white",
        height=260,
        margin=dict(t=50, b=10, l=10, r=10),
    )
    return fig

In [10]:
fig95 = make_pvalue_table(var95_df, "VaR 95% Backtesting Results")

fig95.write_html("../outputs/figures/04_pvalue_table_95.html")

fig95.show()
print("Table saved to outputs/figures/")

Table saved to outputs/figures/


In [11]:
fig99 = make_pvalue_table(var99_df, "VaR 99% Backtesting Results")

fig99.write_html("../outputs/figures/04_pvalue_table_99.html")

fig99.show()
print("Table saved to outputs/figures/")

Table saved to outputs/figures/


In [20]:
# Plot VaR against actual returns for all assets

fig = make_subplots(
    rows=len(TICKERS), cols=1,
    shared_xaxes=True,
    subplot_titles=TICKERS,
    vertical_spacing=0.06,
)

colors_assets = dict(zip(TICKERS, px.colors.qualitative.Plotly))

for i, ticker in enumerate(TICKERS):
    actual  = all_results[ticker]["return"]
    var99   = pd.Series(var_results[0.99][ticker], index=actual.index)
    var95   = pd.Series(var_results[0.95][ticker], index=actual.index)
    exceed_mask = actual.values < var_results[0.99][ticker]

    # Actual returns
    fig.add_trace(go.Scatter(
        x=actual.index, y=actual.values,
        mode="lines", name=f"{ticker} returns",
        line=dict(color=colors_assets[ticker], width=0.7),
        showlegend=False,
    ), row=i+1, col=1)

    # VaR 99%
    fig.add_trace(go.Scatter(
        x=var99.index, y=var99.values,
        mode="lines",
        name="VaR 99%" if i == 0 else None,
        showlegend=(i == 0),
        line=dict(color="#e63946", width=1.5),
    ), row=i+1, col=1)

    # VaR 95%
    fig.add_trace(go.Scatter(
        x=var95.index, y=var95.values,
        mode="lines",
        name="VaR 95%" if i == 0 else None,
        showlegend=(i == 0),
        line=dict(color="#c47000", width=1.2, dash="dot"), #f4a261
    ), row=i+1, col=1)

    # Exceedances
    exc_dates  = actual.index[exceed_mask]
    exc_values = actual.values[exceed_mask]
    fig.add_trace(go.Scatter(
        x=exc_dates, y=exc_values,
        mode="markers",
        name="Exceedance" if i == 0 else None,
        showlegend=(i == 0),
        marker=dict(color="#040720", size=5, symbol="x"),
    ), row=i+1, col=1)

    fig.update_yaxes(title_text="Log return", row=i+1, col=1)

fig.update_layout(
    title="Actual Returns vs VaR 99% and VaR 95% - All Assets",
    height=1000,
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.04),
)
fig.write_html("../outputs/figures/04_var_vs_returns.html")
fig.show()
print("Figure saved to outputs/figures/04_var_vs_returns.html")

Figure saved to outputs/figures/04_var_vs_returns.html


In [22]:
# CVaR Summary table

cvar_rows = []
for ticker in TICKERS:
    cvar_rows.append({
        "Ticker":       ticker,
        "CVaR 99% (mean)": round(
            float(np.mean(cvar_results[0.99][ticker])) * 100, 4),
        "CVaR 95% (mean)": round(
            float(np.mean(cvar_results[0.95][ticker])) * 100, 4),
        "CVaR 99% (worst)": round(
            float(np.min(cvar_results[0.99][ticker])) * 100, 4),
        "CVaR 95% (worst)": round(
            float(np.min(cvar_results[0.95][ticker])) * 100, 4),
    })

cvar_df = pd.DataFrame(cvar_rows).set_index("Ticker")
print("CVaR Summary (% log returns —> negative = loss threshold)\n")
print(cvar_df.to_string())
print("\nNote: CVaR is always more negative than VaR --")
print("it represents the average loss in the worst scenarios, not just the threshold.")

CVaR Summary (% log returns —> negative = loss threshold)

          CVaR 99% (mean)  CVaR 95% (mean)  CVaR 99% (worst)  CVaR 95% (worst)
Ticker                                                                        
AAPL              -6.5704          -4.1922          -29.4690          -18.8025
EURUSD=X          -1.6355          -1.1360           -3.8111           -2.6471
GLD               -3.1849          -2.1765          -10.8949           -7.4456
TIP               -1.2769          -0.8944           -2.7930           -1.9563
^GSPC             -3.7906          -2.5572          -16.5458          -11.1620

Note: CVaR is always more negative than VaR --
it represents the average loss in the worst scenarios, not just the threshold.


In [23]:
# Save VaR series
for level in LEVELS:
    var_df = pd.DataFrame(
        var_results[level],
        index=innovations.index,
    )
    lbl = str(int(level * 100))
    save_processed(var_df,  f"var{lbl}.parquet")

    cvar_df_out = pd.DataFrame(
        cvar_results[level],
        index=innovations.index,
    )
    save_processed(cvar_df_out, f"cvar{lbl}.parquet")

# Save exceedance results table
save_processed(results_df, "exceedance_results.parquet")

print("Saved: var99, var95, cvar99, cvar95, exceedance_results")

Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\var99.parquet
Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\cvar99.parquet
Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\var95.parquet
Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\cvar95.parquet
Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\exceedance_results.parquet
Saved: var99, var95, cvar99, cvar95, exc

In [24]:
# Verification

var99_check  = load_processed("var99.parquet")
results_check = load_processed("exceedance_results.parquet")

assert var99_check.shape == (1000, 5), f"Shape mismatch: {var99_check.shape}"
assert set(var99_check.columns) == set(TICKERS), "Ticker mismatch"
assert (var99_check < 0).all().all(), \
    "VaR values must all be negative (they represent losses)"

print("Verification PASSED")
print(f"\nVaR 99% shape: {var99_check.shape}")
print(f"\nAll VaR values negative: {(var99_check < 0).all().all()}")
print(f"\nExceedance results shape: {results_check.shape}")
print(f"\nFull results summary:")
print(results_check[["Ticker","VaR level","Expected",
                      "Exceedances","p-value","Result"]].to_string(index=False))

Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\var99.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\exceedance_results.parquet  shape=(10, 7)
Verification PASSED

VaR 99% shape: (1000, 5)

All VaR values negative: True

Exceedance results shape: (10, 7)

Full results summary:
  Ticker VaR level  Expected  Exceedances  p-value Result
    AAPL       99%        10            7   0.4264  GREEN
    AAPL       95%        50           48   0.8278  GREEN
EURUSD=X       99%        10            8   0.6343  GREEN
EURUSD=X       95%        50           54   0.5611  GREEN
     GLD       99%        10           10   1.0000  GREEN
     GLD       95%        50           53   0.6629  GREEN
     TIP       99%        10  

----

## Handoff to notebook 05

Saved to `data/processed/`:
- `var99.parquet` -- daily VaR 99% series, all assets
- `var95.parquet` -- daily VaR 95% series, all assets
- `cvar99.parquet` -- daily CVaR 99% series, all assets
- `cvar95.parquet` -- daily CVaR 95% series, all assets
- `exceedance_results.parquet` -- p-value table with color codes

**Notebook 05** completes the full assessment suite:
Kolmogorov-Smirnov, Anderson-Darling, and Christoffersen tests
across all assets, producing the final master results table
for inclusion in the written report.